In [ ]:
import pyvips
import pandas as pd
import scanpy as sc
import numpy as np
import squidpy as sq
import sys
from pathlib import Path
import anndata as ad
import squidpy as sq
from tqdm.notebook import tqdm
import json
import shutil
sys.path.append('../')
from src.utils import preprocess_adata
from src.preprocess_utils.preprocess_image import get_low_res_image
from src.utils import create_cross_validation_folds

In [ ]:
def create_xenium_adata(data_path, drop_border_px=300):
    data = pd.read_parquet(data_path)
    to_keep = data.qv >= 20
    print("Features with QV < 20: ", (~to_keep).sum(), (~to_keep).sum() / len(data))
    data = data[to_keep]
    data.cell_id = data.cell_id.astype(str)
    to_remove = data.cell_id.isin(["UNASSIGNED", "0"])
    print("Unassigned features: ", to_remove.sum())
    data = data[~to_remove].copy()
    if isinstance(data.feature_name.values[0], bytes):
        data.feature_name = data.feature_name.apply(lambda x: x.decode('utf-8'))

    expression_df = data.groupby(["cell_id", "feature_name"]).size()
    expression_df = expression_df.reset_index(name='count')
    expression_df = expression_df.set_index("cell_id")
    expression_df = expression_df.pivot_table(index="cell_id", columns="feature_name", values="count", aggfunc="sum", fill_value=0)

    coord = data.groupby(["cell_id"])[["he_x", "he_y"]].agg("mean")
    coord.columns = ["x_pixel", "y_pixel"]

    adata = ad.AnnData(expression_df.values, obs=coord)
    adata.var_names = expression_df.columns.values
    return adata

In [ ]:
df = pd.read_csv("/cluster/home/knonchev/code/projects2024-cell-embeddings/data/metadata/hg38_gtf.csv")
df = df[~df.gene_name.isna()]
df = df[~df.gene_name.duplicated()]
gene_name = df.gene_name.values  # protein coding genes
gene_name

In [ ]:
dataset_of_interest = ['Lung_Xenium']

In [ ]:
df = pd.read_csv('data/HEST_v1_1_0.csv')
df = df[df.st_technology == "Xenium"]
dataset_count = df.dataset_title.value_counts().sort_values()
dataset_count = dataset_count[dataset_count > 3]
df = df[df.dataset_title.isin(dataset_count.index)]
df["dataset_name_custom"] = df.tissue + "_" + df.st_technology
df

In [ ]:
def create_yaml_file(path, YAML_TEXT):
    with open(path, 'w') as f:
            f.write(YAML_TEXT)


def create_folder(newpath):
    if not os.path.exists(newpath):
        os.makedirs(newpath)

In [ ]:
base_path = "../"
downsample_factor = 10
dot_size = 20

In [ ]:
all_oncotree_code_cluster = []
for oncotree_code in tqdm(df.dataset_name_custom.unique()):
    subset_hest = df[df.dataset_name_custom == oncotree_code]
    patient_ids = subset_hest.patient
    patient_ids = patient_ids[~patient_ids.isna()].values
    unique_patient = np.unique(patient_ids)
    
    subset_hest['groups'] = subset_hest.patient.values
    
    print(f"{oncotree_code}: {len(subset_hest)}")
    unique_patients = list(set([p for p in subset_hest['groups']]))
    patient_replicate_pairs = {p:[s for s in subset_hest.query(f'groups == "{p}"').id.values] for p in unique_patients}
    folds = create_cross_validation_folds(patient_replicate_pairs)

    Path(f"{base_path}/{oncotree_code}").mkdir(parents=True, exist_ok=True)
    Path(f"{base_path}/{oncotree_code}/data").mkdir(parents=True, exist_ok=True)
    Path(f"{base_path}/{oncotree_code}/data/h5ad").mkdir(parents=True, exist_ok=True)
    Path(f"{base_path}/{oncotree_code}/data/image").mkdir(parents=True, exist_ok=True)
    Path(f"{base_path}/{oncotree_code}/data/meta").mkdir(parents=True, exist_ok=True)
    
    import yaml
    # remove references
    yaml.Dumper.ignore_aliases = lambda *args : True
    with open(f'{base_path}/{oncotree_code}/cross_validation_config.yaml', 'w+') as ff:
        yaml.dump(folds, ff, default_flow_style=False)
    
    all_adatas = []

    genes = [create_xenium_adata(f"data/HEST_XENIUM/transcripts/{row.id}_transcripts.parquet").var_names.values for _, row in subset_hest.iterrows()]
    shared_genes = set(genes[0]).intersection(*genes[1:])
    shared_genes = np.array([g for g in shared_genes if g in gene_name]) # shared and protein coding
    
    for _, row in subset_hest.iterrows():
        sample_id = row.id
        image_filename = row.image_filename
        spot_diameter = row.spot_diameter
        group = row.groups
        adata_path = f"data/HEST_XENIUM/transcripts/{sample_id}_transcripts.parquet"
        adata = create_xenium_adata(adata_path)
        adata = adata[:, shared_genes].copy()
        
        if type(adata.X) == np.ndarray:
            pass
        else:
            adata.X = adata.X.toarray()
        sc.pp.filter_cells(adata, min_genes=5)
        sc.pp.filter_cells(adata, min_counts=10)
        adata = preprocess_adata(adata, run_dim_red=False)
        adata.obs.index = [f"{i}_{sample_id}_{oncotree_code}" for i in adata.obs.index]
        adata.obs['batch'] = group
        all_adatas.append(adata)

    all_adatas = ad.concat(all_adatas)
    sc.pp.pca(all_adatas, n_comps=15)
    sc.external.pp.harmony_integrate(all_adatas, key="batch")
    sc.pp.neighbors(all_adatas, use_rep="X_pca_harmony")
    sc.tl.leiden(all_adatas, resolution=0.2)
    all_oncotree_code_cluster.append(all_adatas.obs)

In [ ]:
all_oncotree_code_cluster = pd.concat(all_oncotree_code_cluster)
all_oncotree_code_cluster

In [ ]:
def load_TEMPLATE():
    TEMPLATE = """
SAMPLE:
{}

MODEL:
    - LinearRegressionCell
    - DeepCell
    - MLPCell
    - STNetCell

IMAGE_FEATURES:
    - inception
    - phikon
    - uni
    - resnet50
    - densenet121
    - hoptimus0
    - phikonv2

CELL_RADIUS:
    - 30
    - 60
    - 120

GENE_SET:
    - GO_Biological_Process_2023 
    - GO_Cellular_Component_2023
    - KEGG_2021_Human
    - MSigDB_Hallmark_2020
    - GO_Molecular_Function_2023
    - Reactome_2022

top_n_genes_to_predict: 300
top_n_genes_to_evaluate: 100

DATASET: "HEST"
OUT_FOLDER: "out_benchmark"

DOWNSAMPLE_FACTOR: 10
IMAGE_FORMAT: "tif"

known_genes:

"""
    return TEMPLATE

In [ ]:
for oncotree_code in tqdm(df.dataset_name_custom.unique()):

    subset_hest = df[df.dataset_name_custom == oncotree_code]
    
    patient_ids = subset_hest.patient
    
    sample_string = ""
    for sample in subset_hest.id.values:
        sample_string += f"   - {sample}\n"
    
    text = load_TEMPLATE().format(sample_string)
    create_yaml_file(f'{base_path}/{oncotree_code}/config_dataset.yaml', text)

    genes = [create_xenium_adata(f"data/HEST_XENIUM/transcripts/{row.id}_transcripts.parquet").var_names.values for _, row in subset_hest.iterrows()]
    shared_genes = set(genes[0]).intersection(*genes[1:])
    shared_genes = np.array([g for g in shared_genes if g in gene_name]) # shared and protein coding
    
    for _, row in subset_hest.iterrows():
        sample_id = row.id
        image_filename = row.image_filename
        spot_diameter = row.spot_diameter
    
        adata_path = f"data/HEST_XENIUM/transcripts/{sample_id}_transcripts.parquet"
        adata_out_path = f"{base_path}/{oncotree_code}/data/h5ad/{sample_id}.h5ad"
        image_path = f"data/HEST_XENIUM/wsis/{sample_id}.tif"
        image_out_path = f"{base_path}/{oncotree_code}/data/image/{sample_id}.tif"
        json_path = f"data/HEST_XENIUM/metadata/{sample_id}.json"
        json_out_path = f"{base_path}/{oncotree_code}/data/meta/{sample_id}.json"
    
        json_info = json.load(open(json_path))
    
        adata = create_xenium_adata(adata_path)
        adata = adata[:, shared_genes].copy()

        
        if type(adata.X) == np.ndarray:
            pass
        else:
            adata.X = adata.X.toarray()
        sc.pp.filter_cells(adata, min_genes=5)
        sc.pp.filter_cells(adata, min_counts=10)
        
        cluster_idx = np.array([f"{i}_{sample_id}_{oncotree_code}" for i in adata.obs.index])
        adata.obs['leiden'] = all_oncotree_code_cluster.loc[cluster_idx].leiden.values.astype(str)
        adata.obs['leiden'] = [f"leiden_0.2_{l}" for l in adata.obs['leiden'].values]
        sc.pp.filter_genes(adata, min_counts=1)

            
        json_info['downsample_factor'] = downsample_factor
        json_info['dot_size'] = dot_size
        

        image = pyvips.Image.new_from_file(image_path)
        ### remove cells on the borders
        ### 
        drop_border_px = 200
        x_max = image.width
        y_max = image.height
        
        x_min_threshold = drop_border_px
        x_max_threshold = x_max - drop_border_px
        y_min_threshold = drop_border_px
        y_max_threshold = y_max - drop_border_px
        print(adata.shape)
        adata = adata[
            (adata.obs.x_pixel >= x_min_threshold) & (adata.obs.x_pixel <= x_max_threshold) &
            (adata.obs.y_pixel >= y_min_threshold) & (adata.obs.y_pixel <= y_max_threshold)
        ].copy()
        print(adata.shape)

        x_pixel = adata.obs["x_pixel"]
        y_pixel = adata.obs["y_pixel"]

        adata.obs["x_pixel"] = y_pixel
        adata.obs["y_pixel"] = x_pixel
        
        image = get_low_res_image(image_path, downsample_factor=downsample_factor)
        adata.obsm['spatial'] = adata.obs[["y_pixel", "x_pixel"]].values
        # adjust coordinates to new image dimensions
        adata.obsm['spatial'] = adata.obsm['spatial'] / downsample_factor
        # create 'spatial' entries
        adata.uns['spatial'] = dict()
        adata.uns['spatial']['library_id'] = dict()
        adata.uns['spatial']['library_id']['images'] = dict()
        adata.uns['spatial']['library_id']['images']['hires'] = image

        adata.obs['barcode'] = adata.obs.index
        adata.var["gene_symbol"] = adata.var.index
        
        adata.write_h5ad(adata_out_path)
        
        shutil.copy(image_path, image_out_path)
        with open(json_out_path, 'w') as f:
            f.write(json.dumps(json_info))
    
    